### ETL in Databricks
This notebook gives some basic commands for performing necessary ETL functions. Use this to create a medallion ETL pipeline using the financial dataset.

You should have:
 - bronze schema with tables for each set
 - silver schema with cleaned and formatted tables
 - gold schema with aggregated tables (to answer the questions in the notion page)

The notebook will be used as the source a daily job to refresh the pipeline (The whole notebook will be executed) and a dashboard will be created using the gold tables as source data.


In [0]:
# Project configuration
PROJECT_CATALOG = "demoworkspacejoby"

BRONZE_SCHEMA = "fraud_bronze"
SILVER_SCHEMA = "fraud_silver"
GOLD_SCHEMA = "fraud_gold"

BRONZE_DB = f"{PROJECT_CATALOG}.{BRONZE_SCHEMA}"
SILVER_DB = f"{PROJECT_CATALOG}.{SILVER_SCHEMA}"
GOLD_DB = f"{PROJECT_CATALOG}.{GOLD_SCHEMA}"

BRONZE_VOLUME_PATH = "/Volumes/demoworkspacejoby/default/jarvisvolume/bronze"

In [0]:
# Setup schemas for medallion architecture

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_DB}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_DB}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_DB}")

DataFrame[]

In [0]:
jdbc_url = (
    "jdbc:sqlserver://jarvis-de.database.windows.net:1433;"
    "database=jarvis-de-etl;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

jdbc_user = "add_your_username"

jdbc_password = "add_your_password"

In [ ]:
raw_cards_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "cards_data")
    .option("user", jdbc_user)
    .option("password", jdbc_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

display(raw_cards_df)

In [0]:
raw_cards_df.write.mode("overwrite").saveAsTable(
    f"{BRONZE_DB}.cards_raw"
)

In [ ]:
raw_transactions_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "transactions_data")
    .option("user", jdbc_user)
    .option("password", jdbc_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

display(raw_transactions_df)

In [0]:
raw_transactions_df.write.mode("overwrite").saveAsTable(
    f"{BRONZE_DB}.transactions_raw"
)

In [ ]:
# Read users data from Azure SQL
raw_users_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "users_data")
    .option("user", jdbc_user)
    .option("password", jdbc_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

display(raw_users_df)

In [0]:
# Save users data to Bronze
raw_users_df.write.mode("overwrite").saveAsTable(
    f"{BRONZE_DB}.users_raw"
)

In [0]:
mcc_codes_raw_df = (
    spark.read
    .option("multiline", "true")
    .json(f"{BRONZE_VOLUME_PATH}/mcc_codes.json")
)

fraud_labels_raw_df = (
    spark.read
    .option("multiline", "true")
    .json(f"{BRONZE_VOLUME_PATH}/train_fraud_labels.json")
)

# display(mcc_codes_raw_df)
# display(fraud_labels_raw_df)

In [0]:
# Ingest MCC codes from uploaded JSON file using Python json.load()
# File format expected: {"5812": "Eating Places and Restaurants", ...}

import json
from pyspark.sql.types import StructType, StructField, StringType

mcc_codes_path = f"{BRONZE_VOLUME_PATH}/mcc_codes.json"

with open(mcc_codes_path, "r") as f:
    mcc_raw = json.load(f)

# Convert dictionary to rows
mcc_codes_data = [
    (str(mcc), str(description))
    for mcc, description in mcc_raw.items()
]

# Define clean tabular schema
mcc_schema = StructType([
    StructField("mcc", StringType(), False),
    StructField("mcc_description", StringType(), True)
])

mcc_codes_df = spark.createDataFrame(mcc_codes_data, schema=mcc_schema)

# Save as Bronze table
mcc_codes_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{BRONZE_DB}.mcc_codes_raw"
)

# Validate
mcc_count = spark.table(f"{BRONZE_DB}.mcc_codes_raw").count()

print(f"MCC codes saved: {mcc_count:,} codes")

display(spark.table(f"{BRONZE_DB}.mcc_codes_raw").limit(5))

MCC codes saved: 109 codes


mcc,mcc_description
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees


In [0]:
# Ingest fraud labels from uploaded JSON file using Python json.load()
# File format expected: {"target": {"transaction_id": "Yes"/"No", ...}}

import json
from pyspark.sql.types import StructType, StructField, LongType, IntegerType

fraud_labels_path = f"{BRONZE_VOLUME_PATH}/train_fraud_labels.json"

with open(fraud_labels_path, "r") as f:
    fraud_raw = json.load(f)

# Extract nested target dictionary
target_dict = fraud_raw["target"]

# Convert dictionary to rows
fraud_data = [
    (int(transaction_id), 1 if label == "Yes" else 0)
    for transaction_id, label in target_dict.items()
]

# Define clean tabular schema
fraud_schema = StructType([
    StructField("transaction_id", LongType(), False),
    StructField("is_fraud", IntegerType(), False)
])

fraud_labels_df = spark.createDataFrame(fraud_data, schema=fraud_schema)

# Save as Bronze table
fraud_labels_df.write.mode("overwrite").saveAsTable(
    f"{BRONZE_DB}.fraud_labels_raw"
)

display(spark.table(f"{BRONZE_DB}.fraud_labels_raw").limit(5))

transaction_id,is_fraud
13043699,0
9065802,0
22225988,0
16720107,0
20977432,0


In [0]:
bronze_tables = [
    "cards_raw",
    "transactions_raw",
    "users_raw",
    "mcc_codes_raw",
    "fraud_labels_raw"
]

print("=" * 70)
print("BRONZE INGESTION COMPLETE")
print("=" * 70)

for table_name in bronze_tables:
    full_table_name = f"{BRONZE_DB}.{table_name}"
    row_count = spark.table(full_table_name).count()
    print(f"{full_table_name}: {row_count:,} rows")

print("=" * 70)

BRONZE INGESTION COMPLETE
demoworkspacejoby.fraud_bronze.cards_raw: 6,146 rows
demoworkspacejoby.fraud_bronze.transactions_raw: 220,083 rows
demoworkspacejoby.fraud_bronze.users_raw: 2,000 rows
demoworkspacejoby.fraud_bronze.mcc_codes_raw: 109 rows
demoworkspacejoby.fraud_bronze.fraud_labels_raw: 8,914,963 rows
